# Document Classification Example

Evaluates LLMs on 16-class document classification using [RVL-CDIP](https://huggingface.co/datasets/dvgodoy/rvl_cdip_mini) (OCR-extracted text).

Uses a **QuestionPipeline**: **TemplateQuestionGenerator** → **QuestionRenderer** → **RolloutGenerator**, then scores with `compute_metrics_summary()`.

In [6]:
%pip install lightningrod-ai python-dotenv datasets pandas

from IPython.display import clear_output
clear_output()

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/sign-up?redirect=/api) to get your API key.

- **Google Colab**: Go to the Secrets section (key icon in left sidebar) and add a secret named `LIGHTNINGROD_API_KEY`
- **Local Jupyter**: Set the `LIGHTNINGROD_API_KEY` environment variable, or you'll be prompted to enter it

In [7]:
from dotenv import load_dotenv
from lightningrod import LightningRod
from lightningrod.utils import config

load_dotenv()

api_key = config.get_config_value("LIGHTNINGROD_API_KEY")
lr = LightningRod(api_key=api_key)


## Download and prepare benchmark data

We use the [dvgodoy/rvl_cdip_mini](https://huggingface.co/datasets/dvgodoy/rvl_cdip_mini) dataset from HuggingFace — a 4,000-document subset of the RVL-CDIP benchmark with 16 document types and pre-extracted OCR text.

Each document is converted to a `Sample` with the OCR text as the seed and the ground-truth label stored in metadata for later evaluation.

In [8]:
import random
from datasets import load_dataset
from lightningrod import create_sample

LABEL_NAMES = [
    "letter", "form", "email", "handwritten", "advertisement",
    "scientific_report", "scientific_publication", "specification",
    "file_folder", "news_article", "budget", "invoice",
    "presentation", "questionnaire", "resume", "memo",
]

MIN_TEXT_LENGTH = 50
MAX_DOC_LENGTH = 4000
NUM_SAMPLES = 20

ds = load_dataset("dvgodoy/rvl_cdip_mini", split="test")

# Keep only non-image columns
if "image" in ds.column_names:
    ds = ds.remove_columns(["image"])

random.seed(42)
all_samples = [
    create_sample(seed_text=t[:MAX_DOC_LENGTH], label=LABEL_NAMES[ex["label"]])
    for ex in ds
    if len(t := "\n\n".join(ex.get("ocr_paragraphs") or []).strip()) >= MIN_TEXT_LENGTH
]
samples = random.sample(all_samples, min(NUM_SAMPLES, len(all_samples)))

## Alternative: Load samples from local CSV

Instead of HuggingFace, you can also load samples from a local CSV. Set `LOCAL_CSV_FILE_PATH` to the path of your local file with `seed_text` and `label` columns (can be customized).

In [9]:
LOCAL_CSV_FILE_PATH = config.get_config_value("LOCAL_CSV_FILE_PATH", optional=True)
if LOCAL_CSV_FILE_PATH:
    from lightningrod.preprocessing import file_to_samples

    samples = file_to_samples(LOCAL_CSV_FILE_PATH, csv_seed_text_col="seed_text", csv_label_col="label")
    LABEL_NAMES = sorted(set(s.label.label for s in samples if s.label))
    OPTIONS = {f"option_{i}": name for i, name in enumerate(LABEL_NAMES)}
    print(f"{len(samples)} samples ready for evaluation")

16 samples ready for evaluation


## Upload input dataset

In [10]:
input_dataset = lr.datasets.create_from_samples(samples)
print(f"Created input dataset: {input_dataset.id}")
print(f"Total samples: {input_dataset.num_rows}")

Created input dataset: e0dbee62-299e-4c0d-9afc-55eb9c208f19
Total samples: 16


## Configure the pipeline

The pipeline has three stages:
1. **TemplateQuestionGenerator** — fills a classification prompt template with each document's OCR text
2. **QuestionRenderer** — renders the question with a multiple-choice answer type
3. **RolloutGenerator** — sends the rendered prompt to multiple LLMs via OpenRouter

In [11]:
from lightningrod import (
    QuestionPipeline,
    TemplateQuestionGenerator,
    QuestionRenderer,
    RolloutGenerator,
    MultipleChoiceAnswerType,
    RolloutScorer,
    open_router_model,
)

OPTIONS = {f"option_{i}": name for i, name in enumerate(LABEL_NAMES)}

_options_display = "\n".join(f"{chr(65 + i)}) {name}" for i, name in enumerate(LABEL_NAMES))
QUESTION_TEMPLATE = (
    "What type of document is this? Classify it as one of the following categories:\n\n"
    f"{_options_display}\n\n"
    "Document text:\n{seed_text}"
)

models = [
    open_router_model("openai/gpt-5.2"),
    open_router_model("anthropic/claude-sonnet-4.6"),
    open_router_model("google/gemini-3.1-pro-preview"),
]

answer_type = MultipleChoiceAnswerType()

pipeline = QuestionPipeline(
    question_generator=TemplateQuestionGenerator(question_template=QUESTION_TEMPLATE),
    renderer=QuestionRenderer(answer_type=answer_type),
    rollout_generator=RolloutGenerator(models=models),
    scorer=RolloutScorer(answer_type=answer_type, multiple_choice_options=OPTIONS),
)

## Run the pipeline

This sends each document to all three models for classification. It may take a few minutes depending on the number of samples.

In [12]:
dataset = lr.transforms.run(
    pipeline,
    input_dataset=input_dataset,
    name="Document Classification",
)

Output()

## View results

Download the results and compute per-model accuracy using `compute_metrics_summary()`.

In [13]:
import pandas as pd
from lightningrod.utils import compute_metrics_summary

result_samples = dataset.download()

summary = compute_metrics_summary(result_samples, OPTIONS)
df = pd.DataFrame.from_dict(summary, orient="index")
df.index.name = "model"
df

,accuracy,n_correct,n_parsed,n_total,parse_rate,mean_reward
model,,,,,,
openai/gpt-5.2,0.866667,13,15,16,0.9375,-0.957846
anthropic/claude-sonnet-4.6,0.937500,15,16,16,1.0000,-0.390425
google/gemini-3.1-pro-preview,0.937500,15,16,16,1.0000,-0.135154


## Consensus Results

Does averaging models' probabilities beat any single model?

This analysis shows the accuracy and mean log score of the average of all model predictions vs individual model predictions.

In [14]:
from lightningrod.utils import compute_consensus_summary, compute_multi_choice_consensus

consensus_rows = compute_multi_choice_consensus(result_samples, OPTIONS)
n_agree = sum(1 for r in consensus_rows if r["all_agree"])
print(f"All models agree: {n_agree}/{len(consensus_rows)} ({n_agree/len(consensus_rows):.0%})\n")

summary = compute_consensus_summary(result_samples, OPTIONS)

display(pd.DataFrame.from_dict(summary["model_comparison"], orient="index").rename_axis("model"))


All models agree: 15/16 (94%)



,accuracy,mean_log_score,n
model,,,
averaged,0.937500,-0.237179,16
anthropic/claude-sonnet-4.6,0.937500,-0.390425,16
google/gemini-3.1-pro-preview,0.937500,-0.135154,16
openai/gpt-5.2,0.866667,-0.488369,15
